# Human influence in Rural Wildfires ignitions

## Group: 02
### Name(s) & ID(s) of Group Members:
- Benedita Barreto da Costa Gonçalves - 202205903
- Isabela Moreira Merelles - 202602764
- Maria Leonor Soares Carvalho - 202307522
- Mariana Vale de Oliveira Coelho de Pinho - 202306234

## Table of Contents

1. [Research Question](#1-research-question)
2. [Collect Data](#2-collect-data)
3. [Explore](#3-explore)
4. [Draw Inferences](#4-draw-inferences)
5. [Formulate Conclusions](#5-formulate-conclusions)
6. [Look Back and Ahead](#6-look-back-and-ahead)

---

## 1) Research Question

How did initial response times and burnt areas change before and after 2017, and how are these two variables associated across four Portuguese municipalities?

### Secondary Research Questions:

1. Was the change in the proportion of negligent causes concentrated in the critical summer period, or did it also affect the traditional burning months (autumn/winter)?

    Difference in proportion (Variables to use: cause_type, period, season_risk_period). Test: Z test


2. Did the reduction in negligent wildfire occurrences translate into a significantly lower mean number of daily ignitions on weekends compared to weekdays?

    Mean comparison (Variables to use: Daily_Ignitions_Count, Day_Type, Period). Test: t-test


3. Do final burnt areas differ significantly across the four municipalities under study?

    ANOVA (Variables: Log_Total_Burnt_Area, Municipality). Test: One-way ANOVA.


4. Did the disparity in final burnt area between intentional and negligent causes narrow in the post-2017 period in the municipalities under study?

    ANOVA (Variables to use: Log_Total_Burnt_Area, Cause_Type, Period). Test: Two-way ANOVA 


---

## 2) Collect data

### Dataset Source

The dataset used in this project was sourced from the Institute for Nature Conservation and Forests (ICNF), specifically the wildfire statistics published under the Information Management section of [their website](https://www.icnf.pt/florestas/gfr/gfrgestaoinformacao/estatisticas). This dataset contains data on human influenced wildfire ignitions from 2001 until 2025. The records are split across three ICNF files (2001-2010, 2011-2020 and 2021-2025), and were combined into a single dataset before analysis.

The raw file contains 40 columns covering fire identifiers, location, timestamps, burnt area by land cover, fire weather indices, and cause classification. Only a subset of these columns was used in the analysis, and several variables were derived from them. The full mapping is described below so that the variables can be reproduced from the raw file.

### Dataset Details


#### Variables: 

##### Qualitative Variables

- **Cause_Type**: Intentional vs. Negligent. Derived from the cause classification columns (`TipoCausa`, `GrupoCausa`, `CodCausa`), keeping only validated human causes. Fires with an unclassified or unknown cause were excluded.

- **Period**: Pre-2017 (2001–2016) vs. Post-2017 (2018–2025). Derived from `Ano`. In practice, the law was only implemented in 2021, so the post-2017 group mixes pre-law and post-law years (see limitations).

- **Municipality**: Chaves, Montalegre, Ourém, Tomar. Taken from `Concelho`.

- **Season_Risk_Period**: Critical Period vs. Non-Critical Period. Derived from `Mes` and `Dia`, with the critical period defined as 1 June to 30 September.

- **Day_Type**: Weekend vs. Weekday. Derived from the fire date (`Ano`, `Mes`, `Dia`), with Saturday and Sunday classified as weekend.

- **Cause_Type_Period**: Combined factor of `Cause_Type` and `Period`, used for the two-way ANOVA in Q3.

##### Quantitative Variables

- **Daily_Ignitions_Count**: Number of ignitions per municipality per day, computed by counting `Codigo_SGIF` grouped by `date` and `municipality`.

- **Total_Burnt_Area**: Total surface burnt per occurrence, in hectares, from `AreaTotal_ha`.

- **Initial_Response_Time**: Time in minutes between `DataHoraAlerta` and `DataHora_PrimeiraIntervencao`.

- **Fire_Duration**: Total duration from fire alert to final extinction, in hours, from `Duracao_Horas`.

### Data preparation

In [5]:
%pip install openpyxl --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

project_path = Path.cwd()
raw = project_path / "01_dataset/raw"
processed = project_path / "01_dataset/processed"
processed.mkdir(parents=True, exist_ok=True)

In [11]:
#Loading the data
cols_necessarias = [
    'Ano', 'Mes', 'Dia', 'Hora', 'AreaTotal_ha', 'Duracao_Horas',
    'DataHoraAlerta', 'DataHora_PrimeiraIntervencao', 'DataHora_Extincao',
    'Concelho', 'TipoCausa', 'Codigo_SGIF'
]

df1 = pd.read_excel(raw / "Registos_Incendios_SGIF_2021_2025.xlsx", usecols=cols_necessarias)
df2 = pd.read_excel(raw / "Registos_Incendios_SGIF_2011_2020.xlsx", usecols=cols_necessarias)
df3 = pd.read_excel(raw / "Registos_Incendios_SGIF_2001_2010.xlsx", usecols=cols_necessarias)

# confirm the raw files share the same schema
print("df1 vs df2:", df1.columns.equals(df2.columns))
print("df2 vs df3:", df2.columns.equals(df3.columns))

df1 vs df2: True
df2 vs df3: True


Before deriving variables, the sample size was checked by municipality and year to confirm that all four municipalities are represented across the full period.

In [12]:
# Concatenar os 3 ficheiros (2001 a 2025)
df = pd.concat([df1, df2, df3], ignore_index=True)

concelhos = ["Chaves", "Ourém", "Montalegre", "Tomar"]

# Filtrar pelos 4 concelhos e pela janela completa de 2001 a 2025
df_final = df[df["Concelho"].isin(concelhos)].copy()
df_final = df_final[df_final["Ano"].between(2001, 2025)].copy()

print(df_final["Concelho"].value_counts())
print(pd.crosstab(df_final["Concelho"], df_final["Ano"], margins=True))

Concelho
Montalegre    5635
Chaves        3914
Ourém         2691
Tomar         2280
Name: count, dtype: int64
Ano         2001  2002  2003  2004  2005  2006  2007  2008  2009  2010  ...  \
Concelho                                                                ...   
Chaves       259   307   127   188   326   167   130   139   308   127  ...   
Montalegre   262   283   177   194   390   147   238   149   475   177  ...   
Ourém        118   159   116   166   293   188   136   144    99    97  ...   
Tomar         58   115    86   166   186   125   130   108   110    95  ...   
All          697   864   506   714  1195   627   634   540   992   496  ...   

Ano         2017  2018  2019  2020  2021  2022  2023  2024  2025    All  
Concelho                                                                 
Chaves       132    67    66    73    46   105    36    68    94   3914  
Montalegre   314   119   168   134   191   192   117    94   131   5635  
Ourém         96    82    63    97    6

In [13]:
# quantitative variables:

#Initial_Response_Time
for col in ["DataHoraAlerta", "DataHora_PrimeiraIntervencao", "DataHora_Extincao"]:
  df_final[col] = pd.to_datetime(df_final[col], errors="coerce")

df_final["Initial_Response_Time"] = (
    df_final["DataHora_PrimeiraIntervencao"] - df_final["DataHoraAlerta"]
).dt.total_seconds() / 60

#Fire_Duration
df_final["Fire_Duration"] = df_final["Duracao_Horas"]

#Total_Burnt_Area
df_final["Total_Burnt_Area"] = df_final["AreaTotal_ha"]

# Escala logarítmica para cumprir os pressupostos das ANOVAs
df_final["Log_Total_Burnt_Area"] = np.log10(df_final["Total_Burnt_Area"] + 0.001)

In [15]:
#qualitative variables:

#Period
df_final["Period"] = np.where(df_final["Ano"] <= 2016, "Pre-2017",
                       np.where(df_final["Ano"] >= 2018, "Post-2017", "Transition"))

#Season_Risk_Period
df_final["Season_Risk_Period"] = np.where(
    df_final["Mes"].isin([6, 7, 8, 9]),
    "Critical", "Non-Critical"
)

#Day_Type
df_final["Day_Type"] = np.where(
    df_final["DataHoraAlerta"].dt.dayofweek >= 5,
    "Weekend", "Weekday"
)

In [16]:
# 1. Mapear apenas causas humanas apuradas
cause_map = {
    "Negligente": "Negligent",
    "Intencional": "Intentional",
}

df_final["Cause_Type"] = df_final["TipoCausa"].map(cause_map)

df_final["Cause_Type_Period"] = np.where(
    df_final["Cause_Type"].isna(),
    np.nan,
    df_final["Cause_Type"].astype(str) + "_" + df_final["Period"].astype(str),
)

# 2. Contagem diária de ignições negligentes (excluindo 2017) para a Q2
df_final["Date"] = df_final["DataHoraAlerta"].dt.date

df_neg = df_final[
    (df_final["Cause_Type"] == "Negligent")
    & (df_final["Period"] != "Transition")
].copy()

daily = (
    df_neg.groupby(["Concelho", "Date", "Period"])
    .size()
    .reset_index(name="Daily_Ignitions_Count")
)

daily["Day_Type"] = np.where(
    pd.to_datetime(daily["Date"]).dt.dayofweek >= 5, "Weekend", "Weekday"
)

In [13]:
# Count missing values in each column
print(f"\nNumber of missing values for each column/ feature:")
print(df_final.isnull().sum())


Number of missing values for each column/ feature:
Codigo_SGIF                        0
Codigo_ANEPC                       0
Ano                                0
Mes                                0
Dia                                0
Hora                               0
AreaPov_ha                         0
AreaMato_ha                        0
AreaAgric_ha                       0
AreaTotal_ha                       0
ClasseArea                         0
DataHoraAlerta                     0
DataHora_PrimeiraIntervencao      25
DataHora_Extincao                 43
Duracao_Horas                     43
IncSup24horas                      0
DTCCFR                             0
Distrito                           0
Concelho                           0
Freguesia                          0
Local                              0
RNAP                            6566
RNMPF                           6019
X_Militar                          0
Y_Militar                          0
Latitude               

In [17]:
df_final.describe()

,Ano,Mes,Dia,Hora,AreaTotal_ha,DataHoraAlerta,DataHora_PrimeiraIntervencao,DataHora_Extincao,Duracao_Horas,Initial_Response_Time,Fire_Duration,Total_Burnt_Area,Log_Total_Burnt_Area
count,14520.000000,14520.000000,14520.000000,14520.000000,14520.000000,14520,8937,14477,14477.000000,8937.000000,14477.000000,14520.000000,14520.000000
mean,2011.001446,6.891116,15.793595,14.687190,9.087475,2011-07-14 14:38:06.070248,2015-08-30 20:28:02.517623,2011-07-11 13:07:50.770463,2.433111,18.117153,2.433111,9.087475,-0.772434
min,2001.000000,1.000000,1.000000,0.000000,0.000000,2001-02-18 20:45:00,2008-01-20 17:20:00,2001-02-18 23:05:00,0.000000,0.000000,0.000000,0.000000,-3.000000
25%,2005.000000,5.000000,8.000000,12.000000,0.010000,2005-08-25 18:52:30,2011-09-25 13:12:00,2005-08-25 14:57:00,0.616667,9.000000,0.616667,0.010000,-1.958607
50%,2010.000000,8.000000,16.000000,15.000000,0.200000,2010-10-28 16:55:00,2015-04-03 21:30:00,2010-10-18 17:58:00,1.250000,14.000000,1.250000,0.200000,-0.696804
75%,2016.000000,9.000000,23.000000,18.000000,1.130000,2016-09-11 15:47:45,2019-03-26 18:40:00,2016-09-11 15:38:00,2.533333,20.000000,2.533333,1.130000,0.053463
max,2025.000000,12.000000,31.000000,23.000000,6379.000000,2025-11-04 18:49:00,2025-11-04 19:10:00,2025-11-04 19:24:00,244.550000,1456.000000,244.550000,6379.000000,3.804753
std,6.668413,2.635995,8.666318,5.363572,116.325010,NaN,NaN,NaN,6.112195,30.905899,6.112195,116.325010,1.203978


Saving processed data to be used for statistics

In [18]:
df_final.to_excel(processed / "wildfires_processed.xlsx", index=False)
daily.to_excel(processed / "daily_ignitions.xlsx", index=False)

Per question dataset

In [19]:
# Q1: Negligente vs Intencional, Pré vs Pós, Crítico vs Não-Crítico (Z-test)
q1 = df_final[
    (df_final["Cause_Type"].notna()) & (df_final["Period"] != "Transition")
].copy()
q1.to_excel(processed / "q1_cause_period_season.xlsx", index=False)

# Q2: Ignições diárias de causas negligentes, Weekend vs Weekday (t-test)
q2 = daily.dropna(subset=["Date", "Daily_Ignitions_Count"]).copy()
q2.to_excel(processed / "q2_daily_ignitions.xlsx", index=False)

# Q3: Área ardida por Concelho (One-way ANOVA)
q3 = df_final[
    (df_final["Total_Burnt_Area"].notna())
    & (df_final["Period"] != "Transition")
].copy()
q3.to_excel(processed / "q3_burnt_area_municipality.xlsx", index=False)

# Q4: Área ardida por Cause_Type e Period (Two-way ANOVA)
q4 = df_final[
    (df_final["Cause_Type"].notna())
    & (df_final["Period"] != "Transition")
    & (df_final["Total_Burnt_Area"].notna())
].copy()
q4.to_excel(processed / "q4_burnt_area_cause_period.xlsx", index=False)

#### Missing values

| Question | Variables used | rows dropped where |
|---|---|---|
| Q1 | `Cause_Type`, `Period`, `Season_Risk_Period` | `Cause_Type` is missing or `Period == 'Transition'` (2017) |
| Q2 | `Daily_Ignitions_Count`, `Day_Type`, `Period` | Non-negligent causes, date is missing, or `Period == 'Transition'` |
| Q3 | `Log_Total_Burnt_Area`, `Municipality` | `Total_Burnt_Area` is missing or `Period == 'Transition'` |
| Q4 | `Log_Total_Burnt_Area`, `Cause_Type`, `Period` | `Cause_Type` or `Total_Burnt_Area` is missing, or `Period == 'Transition'` |

In [20]:
log = pd.DataFrame({
    "dataset": ["pooled", "q1", "q2", "q3", "q4"],
    "n_rows": [len(df_final), len(q1), len(q2), len(q3), len(q4)],
    "exclusion": [
        "Initial 4-municipality cohort (2001-2025)",
        "Missing Cause_Type and Transition year (2017) excluded",
        "Negligent daily counts (2017 excluded)",
        "Missing Burnt Area and 2017 excluded",
        "Missing Cause_Type, Burnt Area, and 2017 excluded",
    ],
})
log.to_excel(processed / "sample_sizes.xlsx", index=False)
log

,dataset,n_rows,exclusion
0,pooled,14520,Initial 4-municipality cohort (2001-2025)
1,q1,6519,Missing Cause_Type and Transition year (2017) ...
2,q2,2897,Negligent daily counts (2017 excluded)
3,q3,13816,Missing Burnt Area and 2017 excluded
4,q4,6519,"Missing Cause_Type, Burnt Area, and 2017 excluded"


## 6) Look back and ahead

### Limitations of the Dataset

Two limitations affect interpretation and should be kept in mind when reading the results.

First, the Period split is set at 2017, but the law was only implemented in 2021. The post-2017 group therefore mixes years before and after the law took effect, so any change attributed to the post-2017 period cannot be separated from the effect of the law itself without a further split.

Second, Initial_Response_Time and Fire_Duration depend on how consistently timestamps are recorded across municipalities. The alert source (FonteAlerta) also varies, and response time depends partly on who reported the fire. These differences may introduce variation between municipalities that is not caused by fire behaviour, which is relevant to the municipality comparison in Q4.